## AI Extract and Parse Functions for Document Processing

### Load Invoices into Unity Catalog Volume

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [ ]:
import os
import shutil

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the destination Volume
volume_base = f"/Volumes/{catalog_name}/default/genai_lab/Invoices"

# Create the Invoices directory if it doesn't exist
os.makedirs(volume_base, exist_ok=True)

# Files to copy
files = ["Contoso-3.pdf", "Contoso-6.pdf"]

for file in files:

    # File in Invoices folder beside the notebook
    source_path = f"./Invoices/{file}"

    # Destination in UC Volume
    destination_path = f"{volume_base}/{file}"

    shutil.copy(source_path, destination_path)

    print(f"Copied: {source_path}")
    print(f"To:     {destination_path}")

### Use the ai_parse Function

In [ ]:
%sql

WITH parsed_docs AS (

  SELECT
    path,

    ai_parse_document(
      content,
      MAP('version', '2.0')
    ) AS parsed_content

  FROM READ_FILES(
    CONCAT(
      '/Volumes/YOUR_UNITY_CATALOG_NAME/default/genai_lab/Invoices'
    ),
    format => 'binaryFile'
  )

)

SELECT *
FROM parsed_docs;

### Combine with ai_extract Function

In [ ]:
%sql
WITH parsed_docs AS (

  SELECT
    path,

    ai_parse_document(
      content,
      MAP('version', '2.0')
    ) AS parsed_content

  FROM READ_FILES(
    CONCAT(
      '/Volumes/YOUR_UNITY_CATALOG_NAME/default/genai_lab/Invoices'
    ),
    format => 'binaryFile'
  )

)

SELECT
  path,

  ai_extract(
    parsed_content,

    '{
      "invoice_id": {
        "type": "string",
        "description": "Unique invoice number"
      },

      "vendor_name": {
        "type": "string",
        "description": "Name of the company issuing the invoice"
      },

      "vendor_address": {
        "type": "string",
        "description": "Complete address of the vendor"
      },

      "customer_name": {
        "type": "string",
        "description": "Name of the customer being billed"
      },

      "customer_address": {
        "type": "string",
        "description": "Complete billing address of the customer"
      },

      "invoice_date": {
        "type": "string",
        "description": "Invoice date in YYYY-MM-DD format"
      },

      "due_date": {
        "type": "string",
        "description": "Payment due date in YYYY-MM-DD format"
      },

      "balance_due": {
        "type": "number",
        "description": "Outstanding balance due on the invoice"
      },

      "subtotal": {
        "type": "number",
        "description": "Invoice subtotal before tax and shipping"
      },

      "tax_amount": {
        "type": "number",
        "description": "Total tax charged"
      },

      "shipping_amount": {
        "type": "number",
        "description": "Shipping charge"
      },

      "total_amount": {
        "type": "number",
        "description": "Final invoice total including tax and shipping"
      },

      "line_items": {
        "type": "array",
        "description": "Products or services listed on the invoice",
        "items": {
          "type": "object",
          "properties": {
            "item_name": {
              "type": "string",
              "description": "Name or description of the item"
            },
            "quantity": {
              "type": "integer",
              "description": "Quantity purchased"
            },
            "unit_price": {
              "type": "number",
              "description": "Price per unit"
            },
            "amount": {
              "type": "number",
              "description": "Total amount for this line item"
            }
          }
        }
      }
    }',

    MAP(
      'instructions',
      'Extract invoice information exactly as shown in the document. Do not infer missing values.'
    )

  ) AS invoice_data

FROM parsed_docs;

### Create and Populate a Unity Catalog Table

In [ ]:
%sql

CREATE OR REPLACE TABLE default.invoices
USING DELTA
AS

WITH parsed_docs AS (

  SELECT
    path,

    ai_parse_document(
      content,
      MAP('version', '2.0')
    ) AS parsed_content

  FROM READ_FILES(
    '/Volumes/YOUR_UNITY_CATALOG_NAME/default/genai_lab/Invoices',
    format => 'binaryFile'
  )

),

extracted_docs AS (

  SELECT
    path,

    ai_extract(
      parsed_content,

      '{
        "invoice_id": {
          "type": "string",
          "description": "Unique invoice number"
        },

        "vendor_name": {
          "type": "string",
          "description": "Name of the company issuing the invoice"
        },

        "vendor_address": {
          "type": "string",
          "description": "Complete address of the vendor"
        },

        "customer_name": {
          "type": "string",
          "description": "Name of the customer being billed"
        },

        "customer_address": {
          "type": "string",
          "description": "Complete billing address of the customer"
        },

        "invoice_date": {
          "type": "string",
          "description": "Invoice date in YYYY-MM-DD format"
        },

        "due_date": {
          "type": "string",
          "description": "Payment due date in YYYY-MM-DD format"
        },

        "balance_due": {
          "type": "number",
          "description": "Outstanding balance due on the invoice"
        },

        "subtotal": {
          "type": "number",
          "description": "Invoice subtotal before tax and shipping"
        },

        "tax_amount": {
          "type": "number",
          "description": "Total tax charged"
        },

        "shipping_amount": {
          "type": "number",
          "description": "Shipping charge"
        },

        "total_amount": {
          "type": "number",
          "description": "Final invoice total including tax and shipping"
        },

        "line_items": {
          "type": "array",
          "description": "Products or services listed on the invoice",
          "items": {
            "type": "object",
            "properties": {
              "item_name": {
                "type": "string",
                "description": "Name or description of the item"
              },
              "quantity": {
                "type": "integer",
                "description": "Quantity purchased"
              },
              "unit_price": {
                "type": "number",
                "description": "Price per unit"
              },
              "amount": {
                "type": "number",
                "description": "Total amount for this line item"
              }
            }
          }
        }
      }',

      MAP(
        'instructions',
        'Extract invoice information exactly as shown in the document. Do not infer missing values.'
      )

    ) AS invoice_data

  FROM parsed_docs
)

SELECT

  path AS source_file,

  invoice_data:response:invoice_id:value::STRING
    AS invoice_id,

  invoice_data:response:vendor_name:value::STRING
    AS vendor_name,

  invoice_data:response:vendor_address:value::STRING
    AS vendor_address,

  invoice_data:response:customer_name:value::STRING
    AS customer_name,

  invoice_data:response:customer_address:value::STRING
    AS customer_address,

  invoice_data:response:invoice_date:value::DATE
    AS invoice_date,

  invoice_data:response:due_date:value::DATE
    AS due_date,

  invoice_data:response:balance_due:value::DOUBLE
    AS balance_due,

  invoice_data:response:subtotal:value::DOUBLE
    AS subtotal,

  invoice_data:response:tax_amount:value::DOUBLE
    AS tax_amount,

  invoice_data:response:shipping_amount:value::DOUBLE
    AS shipping_amount,

  invoice_data:response:total_amount:value::DOUBLE
    AS total_amount,

  invoice_data:response:line_items
    AS line_items

FROM extracted_docs;

### Query the Table

In [ ]:
%sql

SELECT *
FROM default.invoices;